In [1]:
!pip install -q -U langchain
!pip install -q -U langchain-core
!pip install -q -U langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.9/136.9 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 1.4 MB/s eta 0:00:00


In [3]:
from google.colab import userdata
GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm=ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.2,
    max_output_tokens=256,
)

# Part 1- MessagesPlaceHolder

Chat History আছে

Human : Hi

AI : Hello

Human : Who are you?

AI : I am an AI.

Human : Tell me about ML

এগুলো Prompt-এর মধ্যে dynamically insert করতে হবে।

এই কাজটাই করে

MessagesPlaceholder

In [6]:
from langchain_core.prompts import(
    ChatPromptTemplate,
    MessagesPlaceholder)

In [7]:
prompt=ChatPromptTemplate.from_messages(
[    (
        "system", "You are a helpful Ai teacher"
    ),
    MessagesPlaceholder(
        variable_name="history"
    ),
    (
        "human",
        "{question}"
    )]
)

System

↓

History

↓

Human

↓

LLM

In [8]:
from langchain_core.messages import (
    HumanMessage,AIMessage
)

In [10]:
history=[
    HumanMessage(content="Hi"),
    AIMessage(content="Hello"),
    HumanMessage(content="Who are you?"),
    AIMessage(content="I am an AI.")
]

In [14]:
formatted_prompt=prompt.invoke(
    {"history":history,
     "question":"Explain LLM in one Line"


    }
)
formatted_prompt

ChatPromptValue(messages=[SystemMessage(content='You are a helpful Ai teacher', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Who are you?', additional_kwargs={}, response_metadata={}), AIMessage(content='I am an AI.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Explain LLM in one Line', additional_kwargs={}, response_metadata={})])

In [15]:
response=llm.invoke(formatted_prompt)
print(response)

content='An LLM is a powerful AI system trained on' additional_kwargs={} response_metadata={'finish_reason': 'MAX_TOKENS', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019f6704-e6ec-73b0-aec4-d16becd3b470-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 28, 'output_tokens': 252, 'total_tokens': 280, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 242}}


In [16]:
print(response.content)

An LLM is a powerful AI system trained on


## part-2 Few-Shot Prompting

In [19]:
from langchain_core.prompts import PromptTemplate

In [20]:
example_prompt=PromptTemplate.from_template(
    """English: {english}
      Bangla :{bangla}
    """
)
example_prompt

PromptTemplate(input_variables=['bangla', 'english'], input_types={}, partial_variables={}, template='English: {english}\n      Bangla :{bangla} \n    ')

In [21]:
examples = [

    {
        "english":"Dog",

        "bangla":"কুকুর"
    },

    {
        "english":"Cat",

        "bangla":"বিড়াল"
    }

]

In [22]:
from langchain_core.prompts import FewShotPromptTemplate

In [23]:
fewshot=FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="""
    English:{input}
    Bangla:
    """,
    input_variables=["input"]

)

In [29]:
prompt=fewshot.invoke(
    {
        "input":"How are u?"
    }
)
prompt

StringPromptValue(text='English: Dog\n      Bangla :কুকুর \n    \n\nEnglish: Cat\n      Bangla :বিড়াল \n    \n\n\n    English:How are u?\n    Bangla:\n    ')

In [30]:
response=llm.invoke(prompt)
print(response.content)

কেমন আছেন?




```
examples
        │
        ▼
example_prompt
        │
        ▼
Each example becomes

English : ....
Bangla : ....

        │
        ▼

suffix

English : {input}
Bangla :

        │
        ▼

input = "Good morning"

        │
        ▼

Final Prompt

English : How are you?
Bangla : তুমি কেমন আছো?

English : I love programming.
Bangla : আমি প্রোগ্রামিং ভালোবাসি।

English : Good morning
Bangla :

        │
        ▼

LLM

        │
        ▼

সুপ্রভাত।
```



## Part-3 Partial Prompt

In [31]:
from langchain_core.prompts import PromptTemplate

In [32]:
template=PromptTemplate.from_template(
    """
    Explain {topic} in language : {language}
    """
)

In [33]:
partial=template.partial(
    language="Bangla"
)

In [37]:
prompt=partial.invoke(
    {
        "topic":"DL"
    }
)

In [38]:
print(prompt)

text='\n    Explain DL in language : Bangla\n    '


In [39]:
response=llm.invoke(prompt)
print(response.content)

গভীর শিক্ষা (Deep Learning) হলো মেশিন ল
